# Lab 3: Morphology

Compute the similarity of the *SMTeuroparl* sentence pairs (test set) using lemmas and the Jaccard distance, compare it with the words of Lab 2, and with the gold standard using the Pearson correlation.

Two pipelines are used and kept separate, so that words and lemmas are always compared within the same library:

- **NLTK**: `word_tokenize` → `pos_tag` → `WordNetLemmatizer` (the lemmatizer of the lab slides).
- **spaCy**: `nlp` gives the tokens, POS tags and lemmas in one pass.

## Setup

Import the libraries and download the NLTK packages (tokenizer, POS tagger, WordNet) and the spaCy model. The code for TextServer (FreeLing) is also here, but it is off (`RUN_TEXTSERVER` in the setup): the service has a daily limit and its answers are only partly cached in `common/`.

In [1]:
import sys
from pathlib import Path

# The notebook runs from labs/, so the workspace root is the parent folder.
ROOT = Path.cwd().parent

DATA_DIR = ROOT / "data" / "test_gold"
NLTK_DIR = ROOT / "models" / "nltk_data"
COMMON_DIR = ROOT / "common"

# TextServer (FreeLing) is off: the service has a daily limit, and only part of the answers are cached in common/.
# Set it to True to add TextServer to all the comparisons.
RUN_TEXTSERVER = False

# Notebook libs
import json
from getpass import getpass

import pandas as pd
import nltk
import requests
from nltk.metrics.distance import jaccard_distance
from nltk.stem import WordNetLemmatizer
from scipy.stats import pearsonr
import spacy

# NLTK packages needed by this notebook
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("punkt_tab", download_dir=NLTK_DIR)
nltk.download("averaged_perceptron_tagger_eng", download_dir=NLTK_DIR)
nltk.download("wordnet", download_dir=NLTK_DIR)
nltk.download("omw-1.4", download_dir=NLTK_DIR)

# TextServer client (textserver.py in common/)
sys.path.insert(0, str(COMMON_DIR))
from textserver import TextServer

def textserver_login(service):
    """TextServer with the user and password of credentials.json in common/, or asked for if it is not there."""
    credentials_file = COMMON_DIR / "credentials.json"
    if credentials_file.exists():
        credentials = json.loads(credentials_file.read_text())
    else:
        credentials = {"user": input("TextServer user: "), "password": getpass("TextServer password: ")}
    return TextServer(credentials["user"], credentials["password"], service)

def textserver_cached(service, sentences, cache_file):
    """sentence -> list of tokens given by TextServer, querying only the sentences not in cache_file yet."""
    answers = json.loads(cache_file.read_text()) if cache_file.exists() else {}
    missing = [s for s in sorted(set(sentences)) if s not in answers]
    if missing:
        ts = textserver_login(service)
        for sentence in missing:
            # The server sometimes fails or does not answer, so try up to 3 times.
            for attempt in range(3):
                try:
                    answer = getattr(ts, service)(sentence)
                    break
                except requests.RequestException:
                    if attempt == 2:
                        raise
            # TextServer may split a sentence in two, so join the tokens of all its sentences.
            answers[sentence] = [token for sent in answer for token in sent]
            # Save after every query, so an interrupted run does not lose the answers.
            cache_file.write_text(json.dumps(answers, ensure_ascii=False, indent=1))
    print(f"TextServer {service}: {len(set(sentences))} distinct sentences, queried now: {len(missing)}")
    return answers

# SpaCy model needed by this notebook
if not spacy.util.is_package("en_core_web_sm"):
    spacy.cli.download("en_core_web_sm")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/models/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/models/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/models/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/models/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Data and preprocessing

Lemmatization needs the POS tag of each word (*leaves* is *leaf* as a noun and *leave* as a verb), so each pipeline runs three steps: tokens → POS tags → lemmas.

- **NLTK**: the Penn Treebank tags of `pos_tag` are mapped to the WordNet categories (`NN*` noun, `VB*` verb, `JJ*` adjective, `RB*` adverb) for the WordNet lemmatizer. The other words keep their form as lemma.
- **spaCy**: the lemmas come directly from `nlp`.

Every sentence is stored as a list of `(word, POS tag, lemma)`.

In [2]:
# Load sentence pairs and gold-standard similarity scores
dt = pd.read_csv(DATA_DIR / 'STS.input.SMTeuroparl.txt', sep='\t', header=None)
dt['gs'] = pd.read_csv(DATA_DIR / 'STS.gs.SMTeuroparl.txt', sep='\t', header=None)
print(f"Sentence pairs: {len(dt)}")

# NLTK pipeline.
wnl = WordNetLemmatizer()
penn_to_wordnet = {'NN': 'n', 'VB': 'v', 'JJ': 'a', 'RB': 'r'}

def nltk_tokens(sentence):
    tokens = []
    for word, tag in nltk.pos_tag(nltk.word_tokenize(sentence)):
        pos = penn_to_wordnet.get(tag[:2])
        tokens.append((word, tag, wnl.lemmatize(word, pos) if pos else word))
    return tokens

# spaCy pipeline.
nlp = spacy.load("en_core_web_sm")

def spacy_tokens(sentence):
    return [(t.text, t.pos_, t.lemma_) for t in nlp(sentence)]

# TextServer pipeline (FreeLing): one "morpho" query per distinct sentence, saved in textserver_morpho.json.
# Each token is [word, lemma, tag, POS].
def textserver_tokens(sentence):
    return [(word, tag, lemma) for word, lemma, tag, pos in ts_morpho[sentence]]

pipelines = ['NLTK', 'spaCy']
if RUN_TEXTSERVER:
    ts_morpho = textserver_cached("morpho", list(dt[0]) + list(dt[1]), COMMON_DIR / "textserver_morpho.json")
    pipelines.append('TextServer')
process = {'NLTK': nltk_tokens, 'spaCy': spacy_tokens, 'TextServer': textserver_tokens}
tokens1 = {pipe: [process[pipe](s) for s in dt[0]] for pipe in pipelines}
tokens2 = {pipe: [process[pipe](s) for s in dt[1]] for pipe in pipelines}

# Example: the first sentence with the two pipelines.
print(dt[0][0])
pd.concat({pipe: pd.DataFrame(tokens1[pipe][0], columns=['word', 'POS', 'lemma']) for pipe in pipelines}, axis=1)

Sentence pairs: 459


The leaders have now been given a new chance and let us hope they seize it.


NLTK                 spaCy               
       word  POS   lemma     word    POS   lemma
0       The   DT     The      The    DET     the
1   leaders  NNS  leader  leaders   NOUN  leader
2      have  VBP    have     have    AUX    have
3       now   RB     now      now    ADV     now
4      been  VBN      be     been    AUX      be
5     given  VBN    give    given   VERB    give
6         a   DT       a        a    DET       a
7       new   JJ     new      new    ADJ     new
8    chance   NN  chance   chance   NOUN  chance
9       and   CC     and      and  CCONJ     and
10      let   VB     let      let   VERB     let
11       us  PRP      us       us   PRON      we
12     hope   VB    hope     hope   VERB    hope
13     they  PRP    they     they   PRON    they
14    seize   VB   seize    seize   VERB   seize
15       it  PRP      it       it   PRON      it
16        .    .       .        .  PUNCT       .

## Similarities and Pearson correlation

For each pipeline, the Jaccard similarity of each pair using words and lemmas. To see how much of the change comes from the case, both are also computed lowercased:

- **words → words, lowercased**: effect of the case alone.
- **words, lowercased → lemmas, lowercased**: effect of the lemmatization alone.
- **lemmas**: as given by the library. spaCy lowercases the lemmas (except proper nouns), NLTK does not.

In [3]:
def jaccard(set1, set2):
    # NLTK returns the Jaccard distance, so convert it to similarity.
    return 1 - jaccard_distance(set1, set2)

representations = {
    'words': lambda sentence: {w for w, _, _ in sentence},
    'words, lowercased': lambda sentence: {w.lower() for w, _, _ in sentence},
    'lemmas': lambda sentence: {l for _, _, l in sentence},
    'lemmas, lowercased': lambda sentence: {l.lower() for _, _, l in sentence},
}

for pipe in pipelines:
    for name, items in representations.items():
        dt[f'{pipe} {name}'] = [jaccard(items(x1), items(x2)) for x1, x2 in zip(tokens1[pipe], tokens2[pipe])]

pd.concat({
    'Pearson': pd.DataFrame({pipe: [pearsonr(dt['gs'], dt[f'{pipe} {r}'])[0] for r in representations] for pipe in pipelines}, index=list(representations)),
    'mean similarity': pd.DataFrame({pipe: [dt[f'{pipe} {r}'].mean() for r in representations] for pipe in pipelines}, index=list(representations)),
}, axis=1).round(4)

Pearson         mean similarity        
                      NLTK   spaCy            NLTK   spaCy
words               0.4507  0.4606          0.5002  0.5053
words, lowercased   0.4627  0.4726          0.5269  0.5320
lemmas              0.4494  0.4784          0.5149  0.5506
lemmas, lowercased  0.4612  0.4811          0.5409  0.5572

## Words vs. lemmas, pair by pair

Count the pairs whose similarity increases or decreases with lemmas, in each pipeline, and check how many pairs are repeated in the dataset.

In [4]:
rows = []
for pipe in pipelines:
    diff = dt[f'{pipe} lemmas'] - dt[f'{pipe} words']
    rows.append({'pipeline': pipe, 'lemmas higher': (diff > 0).sum(), 'lemmas lower': (diff < 0).sum(), 'equal': (diff == 0).sum()})
print(pd.DataFrame(rows).set_index('pipeline'))

# Check for repeated sentence pairs in the dataset.
duplicated = dt.duplicated(subset=[0, 1])
print(f"\nRepeated pairs: {duplicated.sum()} of {len(dt)} ({(~duplicated).sum()} distinct)")
print(f"Different first sentences: {dt[0].nunique()}")

          lemmas higher  lemmas lower  equal
pipeline                                    
NLTK                 97            12    350
spaCy               202             6    251

Repeated pairs: 142 of 459 (317 distinct)
Different first sentences: 27


### Largest increases and decreases

Many pairs are repeated or share the same first sentence, so each first sentence is shown once (the one with the largest change).

In [5]:
def show_pair(i, pipe):
    """Print a sentence pair with its gold score, and the word and lemma sets of one pipeline."""
    print(f"Pair {i}  (gold {dt['gs'][i]})")
    print(f"  S1: {dt[0][i]}")
    print(f"  S2: {dt[1][i]}")
    for name in ['words', 'lemmas']:
        set1, set2 = representations[name](tokens1[pipe][i]), representations[name](tokens2[pipe][i])
        print(f"  {pipe} {name} (Jaccard {dt[f'{pipe} {name}'][i]:.3f})")
        print(f"    shared:  {sorted(set1 & set2)}")
        print(f"    only S1: {sorted(set1 - set2)}")
        print(f"    only S2: {sorted(set2 - set1)}")
    print()

for pipe in pipelines:
    x = dt.assign(diff=dt[f'{pipe} lemmas'] - dt[f'{pipe} words'])
    print(f"=== {pipe}: largest increases with lemmas ===\n")
    for i in x.sort_values('diff', ascending=False, kind='stable').drop_duplicates(subset=[0]).index[:2]:
        show_pair(i, pipe)
    print(f"=== {pipe}: largest decreases with lemmas ===\n")
    for i in x.sort_values('diff', kind='stable').drop_duplicates(subset=[0]).index[:2]:
        show_pair(i, pipe)

=== NLTK: largest increases with lemmas ===

Pair 409  (gold 4.0)
  S1: (Parliament adopted the legislative resolution)
  S2: (the Parliament adopts the legislative resolution)
  NLTK words (Jaccard 0.750)
    shared:  ['(', ')', 'Parliament', 'legislative', 'resolution', 'the']
    only S1: ['adopted']
    only S2: ['adopts']
  NLTK lemmas (Jaccard 1.000)
    shared:  ['(', ')', 'Parliament', 'adopt', 'legislative', 'resolution', 'the']
    only S1: []
    only S2: []

Pair 81  (gold 4.75)
  S1: It increases the power of the big countries at the expense of the small countries.
  S2: Has the effect of increasing the power of the large at the expense of smaller countries.
  NLTK words (Jaccard 0.438)
    shared:  ['.', 'at', 'countries', 'expense', 'of', 'power', 'the']
    only S1: ['It', 'big', 'increases', 'small']
    only S2: ['Has', 'effect', 'increasing', 'large', 'smaller']
  NLTK lemmas (Jaccard 0.643)
    shared:  ['.', 'at', 'country', 'expense', 'increase', 'of', 'power', 's

## POS tags and lemmas

The lemma depends on the POS tag. Compare the tags and lemmas of the two pipelines in pair 12 (the largest spaCy increase) and pair 260 (where the case of *Report* changes).

In [6]:
for i in [12, 260]:
    print(f"Pair {i}  (NLTK lemmas {dt['NLTK lemmas'][i]:.3f}, spaCy lemmas {dt['spaCy lemmas'][i]:.3f})")
    for side, tokens in [('S1', tokens1), ('S2', tokens2)]:
        for pipe in pipelines:
            print(f"  {side} {pipe:<10} {tokens[pipe][i]}")
    print()

Pair 12  (NLTK lemmas 0.556, spaCy lemmas 1.000)
  S1 NLTK       [('(', '(', '('), ('Parliament', 'NNP', 'Parliament'), ('adopted', 'VBD', 'adopt'), ('the', 'DT', 'the'), ('legislative', 'JJ', 'legislative'), ('resolution', 'NN', 'resolution'), (')', ')', ')')]
  S1 spaCy      [('(', 'PUNCT', '('), ('Parliament', 'PROPN', 'Parliament'), ('adopted', 'VERB', 'adopt'), ('the', 'DET', 'the'), ('legislative', 'ADJ', 'legislative'), ('resolution', 'NOUN', 'resolution'), (')', 'PUNCT', ')')]
  S2 NLTK       [('(', '(', '('), ('The', 'DT', 'The'), ('Parliament', 'NNP', 'Parliament'), ('adopts', 'NNS', 'adopts'), ('legislative', 'JJ', 'legislative'), ('resolution', 'NN', 'resolution'), (')', ')', ')')]
  S2 spaCy      [('(', 'PUNCT', '('), ('The', 'DET', 'the'), ('Parliament', 'PROPN', 'Parliament'), ('adopts', 'VERB', 'adopt'), ('legislative', 'ADJ', 'legislative'), ('resolution', 'NOUN', 'resolution'), (')', 'PUNCT', ')')]

Pair 260  (NLTK lemmas 1.000, spaCy lemmas 0.778)
  S1 NLTK       [('

## Lemmas and meaning

Lemmatization removes the information carried by inflection, such as tense. Check it on two sentences that describe different events, and compare the mean similarities with the mean gold score.

In [7]:
# Most pairs are near-paraphrases, and lemmas raise the similarities overall.
print(f"Mean gold score: {dt['gs'].mean():.2f} (scale 0-5)")
for pipe in pipelines:
    print(f"{pipe:<10} mean similarity: words {dt[f'{pipe} words'].mean():.2f}, lemmas {dt[f'{pipe} lemmas'].mean():.2f}")

# Lemmatization removes the tense: these two sentences describe different events.
s1, s2 = "The vote took place yesterday.", "The vote will take place tomorrow."
# These two sentences are not in the dataset, so they are also sent to TextServer.
if RUN_TEXTSERVER:
    ts_morpho.update(textserver_cached("morpho", [s1, s2], COMMON_DIR / "textserver_morpho.json"))
print(f"\nS1: {s1}\nS2: {s2}")
for pipe in pipelines:
    x1, x2 = process[pipe](s1), process[pipe](s2)
    for name in ['words', 'lemmas']:
        set1, set2 = representations[name](x1), representations[name](x2)
        print(f"  {pipe:<10} {name:<6} (Jaccard {jaccard(set1, set2):.3f}) shared: {sorted(set1 & set2)}")

Mean gold score: 4.55 (scale 0-5)
NLTK       mean similarity: words 0.50, lemmas 0.51
spaCy      mean similarity: words 0.51, lemmas 0.55

S1: The vote took place yesterday.
S2: The vote will take place tomorrow.
  NLTK       words  (Jaccard 0.444) shared: ['.', 'The', 'place', 'vote']
  NLTK       lemmas (Jaccard 0.625) shared: ['.', 'The', 'place', 'take', 'vote']
  spaCy      words  (Jaccard 0.444) shared: ['.', 'The', 'place', 'vote']
  spaCy      lemmas (Jaccard 0.625) shared: ['.', 'place', 'take', 'the', 'vote']


## Questions

### 1. Which is better: words or lemmas?

| Representation | NLTK | spaCy |
|---|---|---|
| Words (Lab 2) | 0.4507 | 0.4606 |
| Words, lowercased | 0.4627 | 0.4726 |
| Lemmas | 0.4494 | **0.4784** |
| Lemmas, lowercased | 0.4612 | **0.4811** |

(Pearson correlation with the gold standard.)

**It depends on the pipeline: lemmas are better with spaCy, but not with NLTK.**

- **spaCy**: the correlation rises from **0.4606** (words) to **0.4784** (lemmas). The lemma similarity is higher for 202 pairs and lower for only 6.
- **NLTK**: the correlation does not improve, **0.4507** (words) vs **0.4494** (lemmas). Only 97 pairs are higher and 12 are lower.

**Part of the gain is only the case.** Lowercasing the words raises the correlation by 0.0120 with both libraries (0.4507 → 0.4627 and 0.4606 → 0.4726). The spaCy lemmas are already lowercased (except proper nouns), so their gain includes this effect, while the NLTK lemmas keep the original case. Comparing lowercased words with lowercased lemmas isolates the lemmatization: **+0.0085 with spaCy (0.4726 → 0.4811), −0.0015 with NLTK (0.4627 → 0.4612).**

Where lemmas help, they map inflected forms to the same item:

- **Pair 12** (gold 4.75, largest spaCy increase): *"(Parliament **adopted** the legislative resolution)"* vs *"(**The** Parliament **adopts** legislative resolution)"*. With spaCy, *adopted* / *adopts* become *adopt* and *the* / *The* become *the*, so the similarity goes from 0.556 to 1.000.
- **Pair 350** (gold 5.0): *"**Then** perhaps **we** could have **avoided**..."* vs *"**We** could perhaps **then** have **avoided**..."*. The spaCy lemmas reach 1.000 (from 0.636) only because *Then* / *then* and *We* / *we* are merged, which is the case effect.

The NLTK lemmas fail because they depend on the NLTK POS tagger:

- **Pair 12**: NLTK tags *adopts* as a plural noun (`NNS`), so it stays *adopts*, and *The* keeps its capital letter. The NLTK lemma similarity stays at 0.556.
- **Pair 197** (gold 5.0, largest NLTK decrease): *"The fishermen are inactive, tired and disappointed."* vs *"The fishermen inactive, are tired and disappointed."*. The words are identical (1.000), but with the changed word order the tagger gives different tags: *tired* and *disappointed* keep their form in S1 and become *tire* and *disappoint* in S2, so the similarity drops to 0.636.

### 2. Do you think that could perform better for any pair of texts?

**No.** Lemmas are better on average (with spaCy), but not for every pair:

- **POS tagging errors.** The lemma depends on the tag, and the tag on the context. In pair 260 (gold 4.75), *"Van Orden **Report** (A5-0241/2000)"* vs *"**Report** Van Orden (A5-0241/2000)"*, spaCy tags *Report* as a proper noun in S1 (lemma *Report*) and as a common noun in S2 (lemma *report*), so the similarity drops from 1.000 to 0.778. In pair 101, *Members* is a proper noun in *"Members of Parliament"* and keeps its capital letter, so it no longer matches *member*. With NLTK the same happens in pair 197.
- **Loss of meaning.** Lemmatization removes information carried by inflection, such as tense and number. *"The vote **took** place yesterday."* vs *"The vote **will take** place tomorrow."* describe different events, but their similarity rises from 0.444 (words) to 0.625 (lemmas) with both pipelines, because *took* and *take* both become *take*. In SMTeuroparl most pairs are near-paraphrases (mean gold score 4.55 out of 5), so merging forms usually helps, but in a dataset where these differences matter, lemmas could hurt.
- **Higher similarities overall.** spaCy lemmas raise the mean similarity from 0.51 to 0.55, which also makes unrelated pairs look more alike.

The dataset is also very repetitive: 142 of the 459 pairs are repeated, and there are only 27 different first sentences (the second sentences are different translations of the same sentence). So the examples above stand for many pairs, and the cases where lemmas hurt are few.

In addition, both words and lemmas ignore synonyms (*big* / *large* in pair 81 are still different lemmas), word order and sentence structure, so neither can capture semantic similarity for all pairs.